In [3]:
import fastf1

fastf1.Cache.enable_cache('../data/cache')

session = fastf1.get_session(2025, 'Bahrain', 'Q')
session.load()

core           INFO 	Loading data for Bahrain Grand Prix - Qualifying [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['81', '63', '16', '12', '10', '4', '1', '55', '44', '22', '7', '6', '14', '31', '23', '27', '30', '5', '18', '87']


## Explorando session.laps

- Tabela com todas as voltas de todos os pilotos da sessão (31 colunas)
- Colunas-chave para a Análise 1: `Sector1Time`, `Sector2Time`, `Sector3Time`
- `FreshTyre` (pneu novo True/False) — guardar para a análise de estratégia

In [4]:
session.laps.head()

,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,FreshTyre,Team,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate
0,0 days 00:20:02.956000,PIA,81,NaT,1.0,1.0,0 days 00:17:48.888000,NaT,NaT,0 days 00:00:48.132000,...,True,McLaren,0 days 00:17:48.888000,2025-04-12 16:05:23.850,1,NaN,False,,False,False
1,0 days 00:21:34.348000,PIA,81,0 days 00:01:31.392000,2.0,1.0,NaT,NaT,0 days 00:00:29.287000,0 days 00:00:39.300000,...,True,McLaren,0 days 00:20:02.956000,2025-04-12 16:07:37.918,1,NaN,False,,False,True
2,0 days 00:23:35.891000,PIA,81,0 days 00:02:01.543000,3.0,1.0,NaT,0 days 00:23:34.171000,0 days 00:00:38.060000,0 days 00:00:50.877000,...,True,McLaren,0 days 00:21:34.348000,2025-04-12 16:09:09.310,1,NaN,False,,False,False
3,0 days 00:29:30.152000,PIA,81,NaT,4.0,2.0,0 days 00:27:14.444000,NaT,NaT,0 days 00:00:53.751000,...,False,McLaren,0 days 00:23:35.891000,2025-04-12 16:11:10.853,1,NaN,False,,False,False
4,0 days 00:31:29.404000,PIA,81,0 days 00:01:59.252000,5.0,2.0,NaT,0 days 00:31:27.651000,0 days 00:00:31.125000,0 days 00:00:52.778000,...,False,McLaren,0 days 00:29:30.152000,2025-04-12 16:17:05.114,1,NaN,False,,False,False


In [8]:
session.laps.shape

(277, 31)

In [9]:
session.laps['Driver'].unique()

array(['PIA', 'RUS', 'LEC', 'ANT', 'GAS', 'NOR', 'VER', 'SAI', 'HAM',
       'TSU', 'DOO', 'HAD', 'ALO', 'OCO', 'ALB', 'HUL', 'LAW', 'BOR',
       'STR', 'BEA'], dtype=object)

In [6]:
session.laps.pick_fastest()

Time                      0 days 01:22:31.712000
Driver                                       PIA
DriverNumber                                  81
LapTime                   0 days 00:01:29.841000
LapNumber                                   14.0
Stint                                        6.0
PitOutTime                                   NaT
PitInTime                                    NaT
Sector1Time               0 days 00:00:28.784000
Sector2Time               0 days 00:00:38.574000
Sector3Time               0 days 00:00:22.483000
Sector1SessionTime        0 days 01:21:30.655000
Sector2SessionTime        0 days 01:22:09.229000
Sector3SessionTime        0 days 01:22:31.712000
SpeedI1                                    243.0
SpeedI2                                    273.0
SpeedFL                                    286.0
SpeedST                                    314.0
IsPersonalBest                              True
Compound                                    SOFT
TyreLife            

## Explorando session.results
- NaT efeito cascata (quem foi eliminado no Q1 não corre o Q2 e fica NaT e quem foi eliminado no Q2 não corre o Q3)
- Durante a sessão, a pista evolui (borracha acumula na pista (desgaste), carro com menos combustível) ou seja, uma volta no Q3 costuma ser muito mais rápida que no Q1
- Decisão de análise: o melhor setor de cada piloto na sessão mistura momentos diferentes de cada um na pista

In [7]:
session.results[['Abbreviation', 'Position', 'Q1', 'Q2', 'Q3']]

,Abbreviation,Position,Q1,Q2,Q3
81,PIA,1.0,0 days 00:01:31.392000,0 days 00:01:30.454000,0 days 00:01:29.841000
63,RUS,2.0,0 days 00:01:31.494000,0 days 00:01:30.664000,0 days 00:01:30.009000
16,LEC,3.0,0 days 00:01:31.454000,0 days 00:01:30.724000,0 days 00:01:30.175000
12,ANT,4.0,0 days 00:01:31.415000,0 days 00:01:30.716000,0 days 00:01:30.213000
10,GAS,5.0,0 days 00:01:31.462000,0 days 00:01:30.643000,0 days 00:01:30.216000
4,NOR,6.0,0 days 00:01:31.107000,0 days 00:01:30.560000,0 days 00:01:30.267000
1,VER,7.0,0 days 00:01:31.303000,0 days 00:01:31.019000,0 days 00:01:30.423000
55,SAI,8.0,0 days 00:01:31.591000,0 days 00:01:30.844000,0 days 00:01:30.680000
44,HAM,9.0,0 days 00:01:31.219000,0 days 00:01:31.009000,0 days 00:01:30.772000
22,TSU,10.0,0 days 00:01:31.751000,0 days 00:01:31.228000,0 days 00:01:31.303000


In [10]:
lec = session.laps.pick_drivers('LEC')
lec[['LapNumber', 'LapTime', 'Sector1Time', 'Sector2Time', 'Sector3Time', 'Compound', 'FreshTyre', 'Deleted']]

,LapNumber,LapTime,Sector1Time,Sector2Time,Sector3Time,Compound,FreshTyre,Deleted
35,1.0,NaT,NaT,0 days 00:00:56.279000,0 days 00:00:28.045000,SOFT,True,False
36,2.0,0 days 00:01:31.454000,0 days 00:00:28.967000,0 days 00:00:39.606000,0 days 00:00:22.881000,SOFT,True,False
37,3.0,0 days 00:02:01.988000,0 days 00:00:39.487000,0 days 00:00:50.587000,0 days 00:00:31.914000,SOFT,True,False
38,4.0,NaT,NaT,NaT,NaT,SOFT,False,False
39,5.0,NaT,NaT,0 days 00:00:49.772000,0 days 00:00:26.573000,SOFT,True,False
40,6.0,0 days 00:01:31.056000,0 days 00:00:29.132000,0 days 00:00:39.279000,0 days 00:00:22.645000,SOFT,True,False
41,7.0,0 days 00:01:43.568000,0 days 00:00:32.682000,0 days 00:00:43.245000,0 days 00:00:27.641000,SOFT,True,False
42,8.0,NaT,NaT,0 days 00:00:51.019000,0 days 00:00:24.870000,SOFT,True,False
43,9.0,0 days 00:01:30.724000,0 days 00:00:28.984000,0 days 00:00:39.054000,0 days 00:00:22.686000,SOFT,True,False
44,10.0,0 days 00:02:01.128000,0 days 00:00:41.279000,0 days 00:00:48.667000,0 days 00:00:31.182000,SOFT,True,False
